## Imports

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal

In [ ]:
pyfers_path = os.path.abspath('..')

if pyfers_path not in sys.path:
    sys.path.append(pyfers_path)

import pyfers as fers

## Constants

In [ ]:
FERS_INPUT_FILENAME = os.path.abspath("waveform.h5")
FERS_XML_FILENAME = "sim.fersxml"
FERS_OUTPUT_FILENAME = "receiver_results.h5"
FERS_ANTENNA_FILENAME = os.path.abspath("antenna.xml")

## Simulation Parameters

In [ ]:
n_seconds = 10
adc_bits = 16
adc_rate = 150e6

## Radar Parameters

In [ ]:
chirp = fers.Waveform(name='chirp', f_carrier=9e6, type='pulse', power=20, f_sample=150e6, bandwidth=100e6, t_pulse=2e-6)
antenna = fers.Antenna(name='antenna', type='sinc', gain=2, efficiency=1, az_beta=10, el_beta=5, az_gamma=1, el_gamma=1)
clock = fers.Clock(name='clock', frequency=150e6)
transmitter = fers.Transmitter(antenna=antenna, waveform=chirp, clock=clock, f_prf=1000)
receiver = fers.Receiver(name='receiver', antenna=antenna, clock=clock, f_prf=1000, gate=1000, noise_temp=300)

### Axes

In [ ]:
n_pulses = int(transmitter.f_prf * n_seconds)

r_gate = np.linspace(0, receiver.gate, receiver.ns_gate, endpoint=False)
t_gate = fers.range_to_time(r_gate)

### TX Waveform

In [ ]:
f, t, spect = signal.spectrogram(
    chirp.samples,
    fs=adc_rate,
    nperseg=32,
    nfft=64,
    noverlap=15,
    mode='magnitude',
    window='blackman',
    return_onesided=False,
    detrend=False
)

spect = fers.linear_to_dB(np.abs(spect))
spect -= spect.max()

spect = np.fft.fftshift(spect, axes=0)
t *= 1e6
f = np.fft.fftshift(f)/1e6

plt.figure()
plt.hlines(chirp.bandwidth/2e6, t[0], t[-1], linestyles='--', colors='white')
plt.hlines(-chirp.bandwidth/2e6, t[0], t[-1], linestyles='--', colors='white')
plt.pcolormesh(t, f, spect)
plt.title('Spectrogram of TX Waveform')
plt.xlabel('Time [µs]')
plt.ylabel('Frequency [MHz]')
plt.tight_layout()

### Antenna Beam Pattern

In [ ]:
# convert to dBi
G_az_dBi = 10*np.log10(abs(antenna.az_pattern) + 1e-20)
G_el_dBi = 10*np.log10(abs(antenna.el_pattern) + 1e-20)

# calculate the peak gain
assert round(np.max(G_az_dBi), 1) == round(np.max(G_el_dBi), 1)
G_dBi = np.max(G_az_dBi)
print('Antenna Gain:', round(G_dBi, 2), 'dBi')

# normalise the beam pattern
G_az_dBi -= G_dBi
G_el_dBi -= G_dBi

plt.figure()
plt.title("Normalised Antenna Beam Pattern")
plt.plot(np.degrees(antenna.theta), G_az_dBi, label="Azimuth")
plt.plot(np.degrees(antenna.theta), G_el_dBi, label="Elevation", linestyle="--")
plt.xlabel("Angle (degrees)")
plt.ylabel("Normalised Gain (dB)")
plt.ylim(-20, 0)
plt.xlim(-180, 180)
# plt.xticks([-90, -70, -50, -30, -10, 10, 30, 50, 70, 90])
plt.legend()
plt.grid()

plt.figure()
ax = plt.subplot(1, 1, 1, projection='polar')
ax.set_title('Normalised Antenna Beam Pattern')
ax.plot(antenna.theta, G_az_dBi, label="Azimuth")
ax.plot(antenna.theta, G_el_dBi, label="Elevation")
ax.set_theta_zero_location('N')
ax.set_ylim(-20, 0)
ax.set_xlim(-np.pi, np.pi)
ax.set_rlabel_position(292.5)
ax.grid(True)
ax.legend()

## Platform Parameters

In [ ]:
platform = fers.Platform(d=n_seconds, fs=10, altitude=300, velocity=50, depression=-30, squint=0)

### Simulated Non-Ideal Motion

In [ ]:
# # along-track noise
# platform.add_sinusoidal_noise('x', amplitude=1, frequency=1.6/n_seconds, phase=0)

# # cross-track noise
# platform.add_sinusoidal_noise('y', amplitude=2, frequency=1.3/n_seconds, phase=np.pi/4)
# platform.add_sinusoidal_noise('y', amplitude=0.2, frequency=5.5/n_seconds, phase=0)

# # altitude noise
# platform.add_sinusoidal_noise('z', amplitude=5, frequency=0.75/n_seconds, phase=3*np.pi/4)
# platform.add_sinusoidal_noise('z', amplitude=1, frequency=3.2/n_seconds, phase=np.pi/4)

# # azimuth noise
# platform.add_sinusoidal_noise('az', amplitude=1, frequency=2.3/n_seconds, phase=5*np.pi/8)
# platform.add_sinusoidal_noise('az', amplitude=0.7, frequency=1.1/n_seconds, phase=np.pi/8)

# # elevation noise
# platform.add_sinusoidal_noise('el', amplitude=2, frequency=2.7/n_seconds, phase=3*np.pi/8)

### Real Non-Ideal Motion

In [ ]:
# # real non-ideal platform motion
# t_start = 10 # s
# t_end = 20 # s

# # read motion dataframe from file
# df = pd.read_json("/home/darryn/Downloads/drive-download-20241211T094254Z-001/21_12_04_11_10_23.json", orient='records', lines=True, precise_float=True)

# # trim dataframe
# n_offset = 2
# n_start = n_offset + Fs_motion*t_start
# n_end = n_offset + Fs_motion*t_end
# df = df.iloc[n_start:n_end]

# # condition motion data
# t_platform = np.array(df['timestamp'].values, dtype=float)
# t_platform -= t_platform[0]
# t_platform /= 1e9

# x_motion = df['e'].values
# x_motion -= x_motion[0]

# y_motion = df['n'].values
# y_motion -= y_motion[0]

# # line of best fit
# a, b = np.polyfit(x_motion, y_motion, 1)
# cross_track_noise = y_motion - (a*x_motion + b)

# altitude_noise = df['u'].values
# altitude_noise -= np.mean(altitude_noise)

# velocity_noise = df['abs_vel'].values
# velocity_noise -= np.mean(velocity_noise)

# along_track_noise = velocity_noise * t_platform

### Visualise Platform Motion

In [ ]:
plt.figure()
plt.suptitle("Platform Motion")

plt.subplot(321)
plt.plot(platform.t, platform.x)
plt.ylabel('X (m)')
plt.grid()

plt.subplot(322)
plt.plot(platform.t, platform.az)
plt.ylabel('Azimuth (deg)')
plt.grid()

plt.subplot(323)
plt.plot(platform.t, platform.y)
plt.ylabel('Y (m)')
plt.grid()

plt.subplot(324)
plt.plot(platform.t, platform.el)
plt.ylabel('Elevation (deg)')
plt.grid()

plt.subplot(325)
plt.plot(platform.t, platform.z)
plt.ylabel("Z (m)")
plt.xlabel('Time (s)')
plt.grid()

plt.tight_layout()

## Targets

In [ ]:
targets = []
targets.append(fers.FersTarget(name='A', rcs=500, position_waypoints=[fers.FersPositionWaypoint(x=250, y=600, z=1, t=0)]))
# targets.append(fers.FersTarget(name='B', rcs=100, position_waypoints=[fers.FersPositionWaypoint(x=150, y=400, z=1, t=0)]))
# targets.append(fers.FersTarget(name='C', rcs=100, position_waypoints=[fers.FersPositionWaypoint(x=350, y=200, z=1, t=0)]))

## Visualise Scene

In [ ]:
plt.figure()
ax = plt.axes(projection='3d')

# platform path
ax.plot3D(platform.x, platform.y, platform.z, color='black')
ax.scatter3D(platform.x[0], platform.y[0], platform.z[0], marker='o', color='green')
ax.scatter3D(platform.x[-1], platform.y[-1], platform.z[-1], marker='o', color='red')

# targets
cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
for i, target in enumerate(targets):
    ax.scatter3D(
        target.position_waypoints[0].x,
        target.position_waypoints[0].y,
        target.position_waypoints[0].z,
        marker='^',
        color=cycle[i],
        label=(target.name + ' [' + str(target.rcs) + ' dBsm]')
    )

# ax.set_title('FERS Simulation Scene')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_zlabel('z (m)')
plt.legend()
ax.set_aspect('equal')
# ax.view_init(elev=45, azim=-45)
# ax.view_init(elev=270, azim=180)
plt.tight_layout()

## FERS

In [ ]:
fers_xml = fers.FersXMLGenerator(sim_name='pyfers', xml_filename=FERS_XML_FILENAME)

# TODO tidy this up
fers_xml.add_parameters(t_start=0, t_end=(n_seconds-transmitter.t_pri), sim_rate=adc_rate, bits=adc_bits)

fers_xml.from_waveform(chirp, FERS_INPUT_FILENAME)
fers_xml.from_clock(clock)
# fers_xml.add_antenna(name='antenna', pattern='isotropic')
fers_xml.from_antenna(antenna, FERS_ANTENNA_FILENAME)

fers_xml.add_monostatic_radar(
    antenna='antenna',
    timing='clock',
    prf=transmitter.f_prf,
    waveform=chirp.name,
    position_waypoints=platform.position_waypoints,
    rotation_waypoints=platform.rotation_waypoints,
    window_length=fers.range_to_time(receiver.gate), # range gate
    noise_temp=receiver.noise_temp,
    nodirect='false',
    nopropagationloss='false',
    interp='linear'
)

for t in targets:
    fers_xml.add_target(t)

fers_xml.write_xml()
fers_xml.run()
del fers_xml

## Received Data

In [ ]:
rx_matrix = fers.read_hdf5(FERS_OUTPUT_FILENAME)

### Intensity

In [ ]:
to_plot = fers.linear_to_dB(abs(rx_matrix) + 1e-20)
to_plot -= np.max(to_plot)

plt.imshow(
    to_plot,
    aspect='auto',
    vmax=0,
    vmin=-20,
    interpolation='none',
    extent=[0, t_gate[-1]*1e6, 0, n_pulses],
)
plt.colorbar()
plt.xlabel('Fast Time (us)')
plt.ylabel('Pulse Number')
plt.grid()

### Phase

In [ ]:
plt.imshow(
    np.angle(rx_matrix),
    aspect='auto',
    interpolation='none',
    extent=[0, t_gate[-1]*1e6, 0, n_pulses],
)
plt.colorbar()
plt.xlabel('Fast Time (us)')
plt.ylabel('Pulse Number')
plt.grid()

### Noise Power Bug
FERS appears to vary the noise power level based on the power of the received signal i.e., if there is a strong target present the noise power before and after the signal appears to reduce.

In [ ]:
plt.figure()
# plt.plot(t_gate*1e6, rx_matrix[0].real, label='I-start')
# plt.plot(t_gate*1e6, rx_matrix[0].imag, label='Q-start')
plt.plot(t_gate*1e6, abs(rx_matrix[0]), label='Start')
# plt.plot(t_gate*1e6, rx_matrix[n_pulses//2].real, label='I-mid')
# plt.plot(t_gate*1e6, rx_matrix[n_pulses//2].imag, label='Q-mid')
plt.plot(t_gate*1e6, abs(rx_matrix[n_pulses//2]), label='Middle')
plt.legend()
plt.grid()
plt.xlabel('Time (us)')

# TODO check this!
# print('Predicted Noise Density:', np.round(fers.linear_to_dB(receiver.noise_density) + 30, 2), '[dBm/Hz]')
# print('Measured Noise Density:', round(np.mean(2*fers.linear_to_dB(abs(rx_matrix))) + 30 - fers.linear_to_dB(150e6), 2), 'dBm/Hz')